In [4]:
import sys
import os

# Adiciona o diretório pai (raiz do projeto) ao path
sys.path.append(os.path.abspath(os.path.join('..')))

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# Tokenizador

In [60]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = '''Theoretical analysis and computational modeling are important tools for characterizing what 
nervous systems do, determining how they function, and understanding why they operate in particular 
ways. Neuroscience encompasses approaches ranging from molecular and cellular studies to human psychophysics 
and psychology. Theoretical neuroscience encourages cross-talk among these sub-disciplines by constructing 
compact representations of what has been learned, building bridges between different levels of description, 
and identifying unifying concepts and principles. In this book, we present the basic methods used for these 
purposes and discuss examples in which theoretical approaches have yielded insight into nervous system function.'''

In [68]:
input = tokenizer(
    text,
    padding = 'max_length',
    truncation = True,
    max_lenght = 512,
    return_tensors = 'pt'
)

In [69]:
input_ids = input['input_ids']
print(type(input_ids))
print(input_ids.shape)

<class 'torch.Tensor'>
torch.Size([1, 512])


In [71]:
vocab_size = tokenizer.vocab_size
d_model = 512

In [72]:
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=d_model)
input_embedding = embedding_layer(input_ids)

In [74]:
input_embedding.shape

torch.Size([1, 512, 512])

# Positional Enconding

In [14]:
class PosEmbeding(nn.Module):
    def __init__(self, seq, d_model):
        super().__init__()
        self.seq = seq
        self.d_model = d_model

        pe = self.sinusoid_function()
        self.register_buffer('pe', pe)

# Positional Embedding Function
    def sinusoid_function(self):

        pe = torch.zeros((self.seq, self.d_model))
        

        for pos in range(self.seq):
            for i in range(0, self.d_model, 2):
                pe[pos, i] = math.sin(pos / 10_000 ** (i/self.d_model))
                pe[pos, i+1] = math.cos(pos / 10_000 ** (i/self.d_model))

        return pe

# Mecanismo de Atenção

In [ ]:
# Multi-Head-Attention Mecanism
class QKVAttention(nn.Module):
    def __init__(self, d_model, h, s, causal=False):
        super().__init__()

        self.h = h
        self.reduced_dimension = d_model // h
        self.seq = s
        self.d_model = d_model
        self.batch = 1 # Posso tornar isso dinâmico
        self.causal = causal

        if causal:
            self.register_buffer('mask', torch.tril(torch.ones(s, s)))
        else:
            self.mask = None

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):

        Q, K, V = self.W_Q(x), self.W_K(x), self.W_V(x)

        Q_i = Q.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)
        K_i = K.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)
        V_i = V.reshape(self.batch, self.seq, self.h, self.reduced_dimension).transpose(1, 2)

        score = (Q_i @ K_i.transpose(-2, -1)) / math.sqrt(self.reduced_dimension)

        if self.causal:        
            score = score.masked_fill(self.mask == 0, float('-inf'))
   
        weight = F.softmax(score, dim=-1) @ V_i
        weight = weight.transpose(1, 2).contiguous().view(self.batch, self.seq, self.d_model)


        return self.W_O(weight).squeeze(0)
    

class FeedForward(nn.Module):
    def __init__(self, seq, hidden_size, dropout:float=0.1):
        super().__init__()

        self.f1 = nn.Linear(seq, hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.f2 = nn.Linear(hidden_size, seq)
        
    def forward(self, x):
        x = self.f1(x)
        x = F.relu(x) # Considerar usar outra função de ativação
        x = self.dropout(x)
        x = self.f2(x)
        return x

In [118]:
def add_norm(x):
    l1 = nn.Linear(512, 2048)
    l2 = nn.Linear(2048, 512)
    
    activation = F.relu(
        l1(x)
    )

    l2(activation)

    return l2(activation)
    

def lin(x):

    l = nn.Linear(512, 512)


    return l(x)

# Testes

In [104]:
pe = PosEmbeding(512, 512).pe
x = (pe + input_embedding)

In [ ]:
mmha = QKVAttention(512, 4, 512, True)
mha = QKVAttention(512, 4, 512, False)
ff = FeedForward(512, 700)

# Start of Transformer Block
x_1 = mmha.forward(x)
x_2 = add_norm(x_1)

x_3 = mha.forward(x_2)
x_4 = add_norm(x_3)

x_5 = ff.forward(x_4)
x_6 = add_norm(x_5)

# End of Transformer Block
x_7 = lin(x_6)
x_8 = F.softmax(x_7, dim=-1)

In [123]:
x_8

tensor([[0.0019, 0.0019, 0.0020,  ..., 0.0020, 0.0020, 0.0020],
        [0.0019, 0.0019, 0.0020,  ..., 0.0020, 0.0020, 0.0020],
        [0.0019, 0.0019, 0.0020,  ..., 0.0020, 0.0020, 0.0020],
        ...,
        [0.0019, 0.0019, 0.0020,  ..., 0.0020, 0.0020, 0.0020],
        [0.0019, 0.0019, 0.0020,  ..., 0.0020, 0.0020, 0.0020],
        [0.0019, 0.0019, 0.0020,  ..., 0.0020, 0.0020, 0.0020]],
       grad_fn=<SoftmaxBackward0>)

In [125]:
x

tensor([[[ 0.2543,  0.2545,  0.6709,  ...,  1.2130,  0.6260,  0.8046],
         [ 0.0325,  1.5868,  0.1251,  ..., -1.4293,  0.0201,  2.5644],
         [ 1.5046, -0.4304,  0.3904,  ...,  2.1680,  1.0402, -0.4624],
         ...,
         [-0.6408,  1.9775, -1.8906,  ..., -1.1247, -0.9254,  1.3717],
         [ 0.1705,  1.4665, -1.7390,  ..., -1.1247, -0.9253,  1.3717],
         [ 0.1790,  0.5077, -2.4048,  ..., -1.1247, -0.9252,  1.3717]]],
       grad_fn=<AddBackward0>)